In [ ]:
# Redirect stdout/stderr to a file for remote logging
import sys
import os

os.makedirs("/kaggle/working", exist_ok=True)
class Logger(object):
    def __init__(self):
        self.terminal = sys.stdout
        self.log = open("/kaggle/working/log.txt", "a", encoding="utf-8")
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()
    def flush(self):
        self.terminal.flush()
        self.log.flush()

sys.stdout = Logger()
sys.stderr = Logger()
print("📟 Logging initialized. Stderr/Stdout redirected to /kaggle/working/log.txt")

# @title 🔑 Setup Kaggle Environment & Fetch Credentials from Hub
import os
import sys
import json
import subprocess

# Check GPU capability
check_gpu_code = """
import torch
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    print(major)
else:
    print(0)
"""
try:
    res = subprocess.run(["python3", "-c", check_gpu_code], capture_output=True, text=True)
    gpu_major = int(res.stdout.strip())
    if gpu_major > 0 and gpu_major < 7:
        print(f"⚠️ Detected GPU capability {gpu_major}.0 < 7.0 (older GPU like Tesla P100). Hiding CUDA to fallback to CPU safely.")
        os.environ["CUDA_VISIBLE_DEVICES"] = ""
except Exception as e:
    print(f"Warning checking GPU capability: {e}")

print("🔄 Cloning pipeline repository...")
if os.path.exists("pipeline_code"):
    os.system("rm -rf pipeline_code")

# Step 2 repo is public as per config, clone directly
ret = os.system("git clone --quiet https://github.com/lelehoctiengtrung/lele-step2-voicestroke.git pipeline_code")
if ret != 0:
    print("❌ Failed to clone repository!")
    sys.exit(1)

sys.path.append("./pipeline_code/VPS_Steps")
os.makedirs("/kaggle/working/PROJECTS", exist_ok=True)
os.makedirs("pipeline_code/VPS_Steps", exist_ok=True)

# Fetch credentials from Cloudflare Hub dynamically at runtime
print("📡 Fetching credentials from Cloudflare Hub...")
os.system('curl -s "https://lele-orchestrator-hub.comics2909-1.workers.dev/api/credentials?token=fbac8f27fd1833c411f62ef2225a3cc9d50b3333&file=service_account.json" -o pipeline_code/VPS_Steps/service_account.json')
os.system('curl -s "https://lele-orchestrator-hub.comics2909-1.workers.dev/api/credentials?token=fbac8f27fd1833c411f62ef2225a3cc9d50b3333&file=user_oauth2.json" -o pipeline_code/VPS_Steps/user_oauth2.json')
os.system('curl -s "https://lele-orchestrator-hub.comics2909-1.workers.dev/api/credentials?token=fbac8f27fd1833c411f62ef2225a3cc9d50b3333&file=pipeline_config.json" -o pipeline_code/VPS_Steps/pipeline_config.json')

print("✅ Environment setup complete!")
# Real-time Telegram Logger helper
def send_realtime_log():
    try:
        import sys
        sys.path.append("./pipeline_code/VPS_Steps")
        import telegram_notifier as tel
        log_path = "/kaggle/working/log.txt"
        if os.path.exists(log_path):
            with open(log_path, "r", encoding="utf-8") as f:
                lines = f.readlines()
            log_chunk = "".join(lines[-40:])
            # Wrap in HTML pre tag for formatting
            tel.send_message(f"📟 <b>[Kaggle Console Log]</b>:\n<pre>{log_chunk}</pre>")
    except Exception as e:
        print(f"Failed to send realtime log: {e}")
send_realtime_log()

In [ ]:
# @title 📦 Install Dependencies & Setup Stroke Generator
import os
import shutil
import subprocess

print("⚙️ Installing python dependencies...")
os.system("pip install -q --timeout 120 omnivoice soundfile gspread google-auth google-api-python-client requests urllib3 python-dotenv")

print("⚙️ Checking system dependencies (ffmpeg, chrome)...")
ffmpeg_available = shutil.which("ffmpeg") is not None
if not ffmpeg_available:
    print("  Installing ffmpeg...")
    os.system("apt-get update -y -q && apt-get install -y -q ffmpeg")
else:
    print("  ffmpeg is already installed.")

chrome_path = shutil.which("google-chrome-stable") or shutil.which("google-chrome") or shutil.which("chromium-browser") or "/usr/bin/google-chrome"
if os.path.exists(chrome_path) or shutil.which(chrome_path):
    print(f"  Chrome found at: {chrome_path}. Linking it...")
    os.system(f"ln -sf {chrome_path} /usr/bin/chromium-browser")
else:
    print("  Chrome not found. Downloading and installing...")
    os.system("wget -q --timeout=60 https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb")
    os.system("apt-get update -y -q && apt-get install -y -q ./google-chrome-stable_current_amd64.deb")
    os.system("ln -sf /usr/bin/google-chrome-stable /usr/bin/chromium-browser")
    os.system("rm -f google-chrome-stable_current_amd64.deb")

local_tool_path = "/kaggle/working/Chinese-Stroke-Order-Gen"
repo_tool_path = "./pipeline_code/VPS_Steps/tools/Chinese-Stroke-Order-Gen"

if os.path.exists(repo_tool_path):
    import shutil
    if os.path.exists(local_tool_path):
        shutil.rmtree(local_tool_path)
    shutil.copytree(repo_tool_path, local_tool_path)
    print("✅ Copied Chinese-Stroke-Order-Gen to working directory.")

# Adjust generator settings to use CDN
index_html = os.path.join(local_tool_path, "index.html")
if os.path.exists(index_html):
    with open(index_html, "r", encoding="utf-8") as f:
        content = f.read()
    content = content.replace("/node_modules/hanzi-writer/dist/hanzi-writer.min.js", 
                              "https://cdn.jsdelivr.net/npm/hanzi-writer@2.2/dist/hanzi-writer.min.js")
    with open(index_html, "w", encoding="utf-8") as f:
        f.write(content)

print("⚙️ Running npm install in generator folder...")
try:
    subprocess.run(["npm", "install"], cwd=local_tool_path, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=120)
    print("✅ Stroke Order Generator initialized!")
except subprocess.TimeoutExpired:
    print("⚠️ npm install timed out after 120s! Continuing anyway...")
send_realtime_log()

In [ ]:
# @title 🧠 Initialize OmniVoice Model (Preloaded from Drive & GPU Accelerated)
import os
import tarfile
import torch

# 1. Preload Model Cache from Google Drive to avoid HuggingFace download latency/hangs
local_cache_dir = "/root/.cache/huggingface/hub/models--k2-fsa--OmniVoice"
if not os.path.exists(local_cache_dir):
    os.makedirs(local_cache_dir, exist_ok=True)
    print("📥 Model cache not found. Downloading preloaded model cache from Google Drive...")
    
    # Use gdown to download from user's Drive using the public file ID
    model_drive_id = "19ZtD1QM7ygWoX0NQC3yP25RI7Ug-CB95"
    local_tar_path = "/kaggle/working/omnivoice_model_cache.tar"
    
    ret = os.system(f"gdown --id {model_drive_id} -O {local_tar_path} --quiet")
    if ret == 0 and os.path.exists(local_tar_path):
        print("📦 Extracting model cache tarball...")
        os.system(f"tar -xf {local_tar_path} -C {local_cache_dir}")
        os.remove(local_tar_path)
        print("✅ Model cache preloaded successfully!")
    else:
        print("⚠️ Failed to download via gdown. Fallback to direct HuggingFace download.")

# 2. Initialize OmniVoice
from omnivoice import OmniVoice

device = "cpu"
if torch.cuda.is_available():
    try:
        major, minor = torch.cuda.get_device_capability()
        if major >= 7:
            device = "cuda"
            print(f"GPU capability is {major}.{minor}. Using GPU.")
        else:
            print(f"⚠️ GPU capability is too low ({major}.{minor}) for current PyTorch. Falling back to CPU.")
    except Exception as e:
        print(f"Warning checking GPU capability: {e}. Defaulting to CPU.")
else:
    print("No GPU available. Using CPU.")

dtype = torch.float16 if device == "cuda" else torch.float32
print("📥 Initializing OmniVoice model...")
local_model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map=device, dtype=dtype)
print("✅ OmniVoice Model initialized!")
send_realtime_log()

In [ ]:
try:
# @title 🎙️ Run Step 2 Voice & Stroke Generation
except Exception as e:
    print(f"❌ CRITICAL EXCEPTION IN CELL 4: {e}")
    try:
        send_realtime_log()
    except Exception:
        pass
    raise e